In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

df_uber_cleaned = pd.read_csv ("uber_cleaned.csv")
df_uber_cleaned.head()

,request_id,driver_id,pickup_point,request_timestamp,drop_timestamp,time_slot,date,weekday,status
0,619,1,Airport,11:51:00,13:00:00,Morning,11-07-16,Monday,Trip Completed
1,867,1,Airport,17:57:00,18:47:00,Evening,11-07-16,Monday,Trip Completed
2,1807,1,City,9:17:00,9:58:00,Morning,12-07-16,Tuesday,Trip Completed
3,2532,1,Airport,21:08:00,22:03:00,Night,12-07-16,Tuesday,Trip Completed
4,3112,1,City,8:33:00,9:25:00,Morning,13-07-16,Wednesday,Trip Completed


In [5]:
print("Total rows and columns:", df_uber_cleaned.shape)
print("\nColumn names:", df_uber_cleaned.columns.tolist())

Total rows and columns: (6745, 9)

Column names: ['request_id', 'driver_id', 'pickup_point', 'request_timestamp', 'drop_timestamp', 'time_slot', 'date', 'weekday', 'status']


In [12]:
#1) Requests per Weekday

print(df_uber_cleaned['weekday'].value_counts().sort_index())


Friday       1381
Monday       1367
Thursday     1353
Tuesday      1307
Wednesday    1337
Name: weekday, dtype: int64


In [14]:
#2) Number of Requests by Status

requests_by_status = df_uber_cleaned['status'].value_counts()
print("Number of requests by status:\n", requests_by_status)


Number of requests by status:
 Trip Completed       2831
No Cars Available    2650
Cancelled            1264
Name: status, dtype: int64


In [19]:
#3) Status Distribution by Weekday

ordered_days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

status_by_weekday = (
    df_uber_cleaned.groupby(['weekday', 'status'])
    .size()
    .unstack(fill_value=0)
    .reindex(ordered_days)  # to keep weekday order
)
print("Status distribution by weekday:\n", status_by_weekday)



Status distribution by weekday:
 status     Cancelled  No Cars Available  Trip Completed
weekday                                                
Monday         262.0              504.0           601.0
Tuesday        240.0              505.0           562.0
Wednesday      270.0              490.0           577.0
Thursday       252.0              571.0           530.0
Friday         240.0              580.0           561.0
Saturday         NaN                NaN             NaN
Sunday           NaN                NaN             NaN


In [20]:
#4) Status Distribution by Pickup Point

status_by_pickup_point = df_uber_cleaned.groupby(['pickup_point', 'status']).size().unstack(fill_value=0)
print("Status distribution by pickup point:\n", status_by_pickup_point)


Status distribution by pickup point:
 status        Cancelled  No Cars Available  Trip Completed
pickup_point                                              
Airport             198               1713            1327
City               1066                937            1504


In [27]:
#5) Weekdays with Highest “No Cars Available” Incidents

no_cars_df = df_uber_cleaned[df_uber_cleaned['status'] == 'No Cars Available']
no_cars_by_week = no_cars_df['weekday'].value_counts().sort_values(ascending=False)
print("Weekday with highest 'No Cars Available' incidents:\n", no_cars_by_week)


Weekday with highest 'No Cars Available' incidents:
 Friday       580
Thursday     571
Tuesday      505
Monday       504
Wednesday    490
Name: weekday, dtype: int64


In [29]:
#6) Number of Unique Drivers

unique_drivers = df_uber_cleaned['driver_id'].nunique()
print("Number of unique drivers:", unique_drivers)


Number of unique drivers: 300


In [45]:
#7) Hourly Distribution of Different Statuses (with Time Slots)

df_uber_cleaned['request_timestamp'] = pd.to_datetime(df_uber_cleaned['request_timestamp'])

df_uber_cleaned['hour'] = df_uber_cleaned['request_timestamp'].dt.hour

def get_time_slot(hour):
    if 0<= hour <4:
        return 'Late Night(0-4)'
    elif 5 <= hour < 7:
        return 'Early Morning (5-7)'
    elif 8 <= hour < 11:
        return 'Morning (8-11)'
    elif 12 <= hour < 15:
        return 'Afternoon (12-15)'
    elif 16 <= hour < 19:
        return 'Evening (16-19)'
    else:
        return 'Night (20-23)'


df_uber_cleaned['time_slot'] = df_uber_cleaned['hour'].apply(get_time_slot)

status_hourly_distribution = df_uber_cleaned.groupby(['time_slot', 'status']).size().unstack(fill_value=0)

print("Hourly distribution of request statuses:")
print(status_hourly_distribution)


Hourly distribution of request statuses:
status               Cancelled  No Cars Available  Trip Completed
time_slot                                                        
Afternoon (12-15)           48                134             298
Early Morning (5-7)        321                170             352
Evening (16-19)             81                600             406
Late Night(0-4)             14                225             136
Morning (8-11)             415                238             444
Night (20-23)              385               1283            1195
